#### Code to convert daily detected granules for PRR ocean color to an html file

In [1]:
# imports, make meng's tools available

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys
sys.path.append(r'C:\Users\gtrolley\Documents\GitHub\pace-rapid-response\dust\mapoltool\tools')
from detection_html_all import *
import io
from io import BytesIO
import os

In [2]:
# function definitions
def fig_to_base64(input_source, factor=1, output_format='PNG', quality=85):
    '''function by gt
       A function to manipulate figures made by fig, ax = plt.subplots OR PNG file paths to prepare them for writing
       as base64 encoded strings to html files. Returns a base64 encoded string manipulated 
       with the desired factor and quality. factor must be an integer, factor=2 means size is reduced by 50%
       
       Args:
           input_source: Either a matplotlib figure object or a string path to a PNG file
           factor: Integer resize factor (factor=2 means size is reduced by 50%)
           output_format: 'PNG' or 'JPEG'
           quality: JPEG quality (1-100, ignored for PNG)
    '''
    
    # Check if input is a string (file path) or matplotlib figure
    if isinstance(input_source, str):
        # Handle file path input
        if not os.path.exists(input_source):
            raise FileNotFoundError(f"Image file not found: {input_source}")
        
        # Open the image directly from file
        img = Image.open(input_source)
        #print(f"Loaded image from file: {input_source}")
        original_size = img.size
        
    else:
        # Handle matplotlib figure input (original logic)
        buffer = io.BytesIO()
        input_source.savefig(buffer, format=output_format, dpi=300)
        buffer.seek(0)
        
        # Open image from buffer
        img = Image.open(buffer)
        original_size = img.size
    
    #print(f"Original size: {original_size}")
    
    # Resize image if factor > 1
    if factor > 1:
        new_width = max(1, img.width // factor)
        new_height = max(1, img.height // factor)
        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    
    #print(f"Final size: {img.size}")

    # Save processed image to buffer
    output_buffer = io.BytesIO()
    
    if output_format.upper() == 'JPEG':
        img = img.convert("RGB")  # JPEG doesn't support transparency
        img.save(output_buffer, format=output_format, quality=quality)
    else:
        img.save(output_buffer, format=output_format)  # PNG ignores quality
    
    # Convert to base64
    base64_string = base64.b64encode(output_buffer.getvalue()).decode("utf-8")
    return base64_string

def write_html_image_row(imgs, capts, sizes):
    ''' function by GT
    function to write a row of images to html. provide 3 lists, imgs is list of base64 string images, 
    capts is list of captions, sizes is list of 'small' or 'large. all lists should be same length '''

    f.write("<div class='row'>\n")  # Add the row wrapper with display: flex

    for i in range(len(imgs)):
        f.write(f"<div class='img-container {sizes[i]}'>\n")
        f.write(f"<img src='data:image/jpeg;base64,{imgs[i]}' alt='{capts[i]}' />\n")
        f.write(f"<div class='caption'>{capts[i]}</div>\n</div>\n")

    
    f.write("</div>\n")


In [3]:
# list files in this directory
os.listdir()

['.DS_Store', '20250915', 'daily_OC_HTML.ipynb']

In [4]:
# USER INPUTS:
day = '20250915' # choose the directory from os.listdir above with the day of data you want

In [5]:
ofilepath = day + '/html/OCI_chlor_a_anomaly_daily_'+day+'.html'
granule_folders  = [item for item in os.listdir(day+'/png/') if os.path.isdir(os.path.join(day+'/png/', item))]
header_imgs  = [item for item in os.listdir(day+'/png/') if item.lower().endswith('.png')]

In [6]:
header_imgs

['L3_Chl_20250915.png',
 'L3_Chl_20250915_bboxes.png',
 'L3_Chl_30dayMean_20250915.png',
 'L3_Chl_30dayMean_20250915_bboxes.png',
 'L3_Chl_Anomaly_20250915.png']

In [22]:
# html file anatomy:

# overall header: provide oversight for method, and show: 
# daily vs monthly chla (row 1)
# chla anomaly, fille width figure

# then, granule by granule add figures


h1 = 'PACE OCI daily Chlorophyll-a Anomaly, '+day
tab_title = day+'_chla_anom_oci'
with open(ofilepath, "w") as f:
        # Start the HTML file
        f.write("<!DOCTYPE html>\n<html>\n<head>\n<meta charset='UTF-8'>\n")
        f.write(f"<title>{tab_title}</title>\n")
        f.write("<style>\nbody { font-family: Arial, sans-serif; margin: 20px; }\n")
        f.write(".gallery { display: flex; flex-direction: column; gap: 20px; }\n")
        f.write(".row { display: flex; justify-content: center; gap: 20px; }\n")
        f.write(".small { flex: 1; } /* Shrink the globe */\n")
        f.write(".large { flex: 2; } /* Full-size image */\n")
        f.write(".img-container { display: flex; flex-direction: column; align-items: center; border: 1px solid #ccc; ")
        f.write("padding: 10px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); background-color: #fafafa; }\n")
        f.write(".img-container img { max-width: 100%; height: auto; display: block; border-radius: 5px; }\n")
        f.write(".caption { margin-top: 10px; font-weight: bold; text-align: center; }\n")
        f.write("</style>\n</head>\n<body>\n")
        f.write(f"<h1>{h1}</h1>\n")
        f.write(f"""<p>Daily chlorophyll-a anomaly for PACE-OCI for use informing 
                    PACE Rapid Response Project. Calculated as the daily L3M chl-a image math math math with the L3M Chl-a monthyl mean</p>\n""")

        f.write(f"<div>Developed by Graham Trolley and Matthew Kehrli</div>\n")

        imgs = [fig_to_base64(day+'/png/'+'L3_Chl_'+day+'_bboxes.png'),fig_to_base64(day+'/png/'+'L3_Chl_30dayMean_'+day+'_bboxes.png') ]
        capts = ['Daily Chl_a '+day, 'Chl_a 30-day']
        sizes = ['large', 'large']
        write_html_image_row(imgs, capts, sizes)

        imgs = [fig_to_base64(day+'/png/'+'L3_Chl_Anomaly_'+day+'.png') ]
        capts = ['Chl_a anomaly '+day+' vs 30-day mean']
        sizes = ['large']
        write_html_image_row(imgs, capts, sizes)


        #header is done, now write a loop to plot all the granule data


        granule_images_shrink_factor = 4 # use a scale factor to reduce quality of saved images, greatly aids in keeping filesize reasonable
        for granule in granule_folders:
                f.write(f"<div>______________________________________________________</div>\n")
                f.write(f"<h1>{'Granule '+granule}</h1>\n")

                carbon_phyto_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_carbon_phyto_overlay.png', factor = granule_images_shrink_factor)
                chlor_a_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_chlor_a_overlay.png', factor = granule_images_shrink_factor)
                poc_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_poc_overlay.png', factor = granule_images_shrink_factor)

                imgs = [chlor_a_im,carbon_phyto_im, poc_im ]
                capts = ['','','']
                sizes = ['large', 'large', 'large']
                write_html_image_row(imgs, capts, sizes)


        f.write("</body>\n</html>\n")

In [24]:
day+'/png/'+'L3_Chl_'+day+'_bboxes.png'

'20250915/png/L3_Chl_20250915_bboxes.png'

In [15]:
os.listdir(day+'/png/'+granule_folders[0])

['20250915T003940_carbon_phyto_overlay.png',
 '20250915T003940_chlor_a_overlay.png',
 '20250915T003940_poc_overlay.png']

In [16]:
granule_folders[0]

'20250915T003940'

In [ ]:
day+'/png/'+